# CPI Backfill — UBOS Publications Scraper

Downloads CPI Excel/PDF reports from the UBOS statistical publications page.
Goal: collect maize-specific CPI (whole grain maize + maize flour) going back to 2006
to extend the model training window beyond July 2017.

**After downloading:** open each Excel file, pull the maize CPI columns from the
Decomposed sheet (codes 01.1.1.1.1 and 01.1.1.2.1), and append to `data/maize_cpi_monthly.csv`.

## 0. Config — set your options here

In [4]:
# Which keywords to match in publication titles
KEYWORDS = ["CPI"]

# Only fetch publications from this year and earlier (None = all years)
MAX_YEAR = 2017

# Where to save downloaded files
OUTPUT_DIR = "./ubos_downloads"

# Set True to list matches without downloading
DRY_RUN = True

## 1. Scrape the UBOS publications page

In [5]:
import os
import re
import time
from urllib.parse import urljoin, urlparse

import requests
import urllib3
from bs4 import BeautifulSoup
import pandas as pd

# UBOS uses a cert chain that macOS Python can't verify out of the box.
# Safe to suppress here — this is a read-only scrape of a known government site.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://www.ubos.org/publications/statistical/"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}


def parse_year(date_str: str) -> int | None:
    match = re.search(r"\b(20\d{2}|19\d{2})\b", date_str)
    return int(match.group(1)) if match else None


def scrape_publications(keywords: list[str], max_year: int | None) -> list[dict]:
    print(f"Fetching {BASE_URL} ...")
    resp = requests.get(BASE_URL, headers=HEADERS, timeout=30, verify=False)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    records = []

    for a_tag in soup.find_all("a", href=True):
        title = a_tag.get_text(strip=True)
        href = a_tag["href"]

        if not any(ext in href.lower() for ext in [".pdf", ".xlsx", ".xls", ".zip", ".csv", "upload"]):
            if not any(kw.lower() in title.lower() for kw in keywords):
                continue

        if not any(kw.lower() in title.lower() for kw in keywords):
            continue

        date_str = ""
        row = a_tag.find_parent("tr") or a_tag.find_parent("li") or a_tag.find_parent("div")
        if row:
            row_text = row.get_text(" ", strip=True)
            date_match = re.search(r"\b\d{2}/\d{2}/\d{4}\b", row_text)
            if date_match:
                date_str = date_match.group(0)

        year = parse_year(date_str)

        if max_year and year and year > max_year:
            continue

        full_url = urljoin(BASE_URL, href)
        records.append({"title": title, "url": full_url, "date": date_str, "year": year})

    seen = set()
    unique = [r for r in records if not (r["url"] in seen or seen.add(r["url"]))]
    return unique


records = scrape_publications(KEYWORDS, MAX_YEAR)
print(f"\nFound {len(records)} matching publications")

Fetching https://www.ubos.org/publications/statistical/ ...

Found 20 matching publications


## 2. Preview what was found

In [6]:
df = pd.DataFrame(records).sort_values("year", ascending=False)
pd.set_option("display.max_colwidth", 80)
display(df[["year", "date", "title", "url"]])

,year,date,title,url
1,2017.0,29/12/2017,CPI December,https://www.ubos.org/wp-content/uploads/publications/CPI Publication for Dec...
8,2017.0,31/05/2017,CPI May,https://www.ubos.org/wp-content/uploads/publications/CPI Publication for May...
12,2017.0,24/01/2017,CPI January 2017,https://www.ubos.org/wp-content/uploads/publications/CPI Publication for Jan...
11,2017.0,28/02/2017,CPI February 2017,https://www.ubos.org/wp-content/uploads/publications/CPI Publication for Feb...
2,2017.0,29/12/2017,CPI July,https://www.ubos.org/wp-content/uploads/publications/CPI Publication for Jul...
9,2017.0,28/04/2017,CPI April,https://www.ubos.org/wp-content/uploads/publications/CPI Publication for Apr...
10,2017.0,31/03/2017,CPI March,https://www.ubos.org/wp-content/uploads/publications/CPI Publication for Mar...
7,2017.0,30/06/2017,CPI June,https://www.ubos.org/wp-content/uploads/publications/CPI press Release June ...
6,2017.0,20/07/2017,CPI August,https://www.ubos.org/wp-content/uploads/publications/CPI Publication for Aug...
5,2017.0,27/09/2017,CPI September,https://www.ubos.org/wp-content/uploads/publications/CPI Publication for Sep...


## 3. Download

Set `DRY_RUN = False` in cell 0 and re-run when you're ready to download.

In [ ]:
def safe_filename(title: str, url: str) -> str:
    ext = os.path.splitext(urlparse(url).path)[-1] or ".pdf"
    name = re.sub(r'[\\/*?:"<>|]', "_", title)
    name = re.sub(r"\s+", "_", name).strip("_")
    return f"{name}{ext}"[:180]


def download_file(record: dict, output_dir: str, session: requests.Session) -> str:
    """Returns 'ok', 'skip', or 'fail'."""
    filename = safe_filename(record["title"], record["url"])
    filepath = os.path.join(output_dir, filename)

    if os.path.exists(filepath):
        print(f"  [skip] {filename}")
        return "skip"

    try:
        resp = session.get(record["url"], headers=HEADERS, timeout=60, stream=True, verify=False)
        resp.raise_for_status()
        with open(filepath, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  [ok]   {filename}  ({size_kb:.0f} KB)")
        return "ok"
    except Exception as e:
        print(f"  [fail] {filename}: {e}")
        return "fail"


if DRY_RUN:
    print("DRY_RUN=True — set it to False in cell 0 to download.")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Downloading {len(records)} files to {os.path.abspath(OUTPUT_DIR)}\n")
    session = requests.Session()
    results = {"ok": 0, "skip": 0, "fail": 0}

    for r in records:
        status = download_file(r, OUTPUT_DIR, session)
        results[status] += 1
        time.sleep(0.5)  # polite crawl delay

    print(f"\nDone — {results['ok']} downloaded, {results['skip']} skipped, {results['fail']} failed.")

## 4. Inspect downloaded files

In [ ]:
if os.path.isdir(OUTPUT_DIR):
    files = sorted(os.listdir(OUTPUT_DIR))
    print(f"{len(files)} files in {OUTPUT_DIR}:")
    for f in files:
        size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
        print(f"  {size_kb:6.0f} KB  {f}")
else:
    print("Output directory does not exist yet — run the download cell first.")